In [1]:
import pandas as pd
import numpy as np
import os

# Set the path to the data folder
data_path = '../data/raw/'

# Load the Olympic athlete events data
print("Loading Olympic athlete events data...")
athletes = pd.read_csv(data_path + 'athlete_events.csv')
print(f"Loaded {len(athletes)} athlete-event records")
print(f"Columns: {list(athletes.columns)}")

# Load the NOC regions mapping
print("\nLoading NOC regions mapping...")
noc_map = pd.read_csv(data_path + 'noc_regions.csv')
print(f"Loaded {len(noc_map)} NOC region mappings")
print(f"Columns: {list(noc_map.columns)}")

# Load the GDP data with special handling for World Bank format
print("\nLoading GDP data...")
# World Bank files often have metadata rows, so we skip them
gdp_raw = pd.read_csv(data_path + 'API_NY.GDP.PCAP.CD_DS2_en_csv_v2_31.csv', 
                      skiprows=4)
print(f"Loaded {len(gdp_raw)} country records")
print(f"Columns (first 10): {list(gdp_raw.columns[:10])}")
print(f"Columns (last 5): {list(gdp_raw.columns[-5:])}")

Loading Olympic athlete events data...
Loaded 271116 athlete-event records
Columns: ['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games', 'Year', 'Season', 'City', 'Sport', 'Event', 'Medal']

Loading NOC regions mapping...
Loaded 230 NOC region mappings
Columns: ['NOC', 'region', 'notes']

Loading GDP data...
Loaded 266 country records
Columns (first 10): ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965']
Columns (last 5): ['2021', '2022', '2023', '2024', 'Unnamed: 69']


In [2]:
# Check the first few rows of each dataset to understand structure
print("=== OLYMPIC DATA - First 3 rows ===")
print(athletes.head(3))

print("\n=== NOC REGIONS - First 5 rows ===")
print(noc_map.head(5))

print("\n=== GDP DATA - First 3 rows ===")
print(gdp_raw.head(3))

# Check for missing values in key columns
print("\n=== Missing Values in Olympic Data ===")
print(athletes[['NOC', 'Year', 'Medal']].isnull().sum())

print("\n=== Missing Values in NOC Mapping ===")
print(noc_map.isnull().sum())

=== OLYMPIC DATA - First 3 rows ===
   ID                 Name Sex   Age  Height  Weight     Team  NOC  \
0   1            A Dijiang   M  24.0   180.0    80.0    China  CHN   
1   2             A Lamusi   M  23.0   170.0    60.0    China  CHN   
2   3  Gunnar Nielsen Aaby   M  24.0     NaN     NaN  Denmark  DEN   

         Games  Year  Season       City       Sport  \
0  1992 Summer  1992  Summer  Barcelona  Basketball   
1  2012 Summer  2012  Summer     London        Judo   
2  1920 Summer  1920  Summer  Antwerpen    Football   

                          Event Medal  
0   Basketball Men's Basketball   NaN  
1  Judo Men's Extra-Lightweight   NaN  
2       Football Men's Football   NaN  

=== NOC REGIONS - First 5 rows ===
   NOC       region                 notes
0  AFG  Afghanistan                   NaN
1  AHO      Curacao  Netherlands Antilles
2  ALB      Albania                   NaN
3  ALG      Algeria                   NaN
4  AND      Andorra                   NaN

=== GDP DATA 

In [3]:
# Create a copy to work with
olympic_clean = athletes.copy()

# Handle the Medal column - convert 'NA' string to actual NaN
# Then create a binary column for whether athlete won any medal
olympic_clean['Medal_Won'] = olympic_clean['Medal'].notna()

# Create a numeric medal value for analysis
# Gold=3, Silver=2, Bronze=1, No medal=0
medal_map = {'Gold': 3, 'Silver': 2, 'Bronze': 1}
olympic_clean['Medal_Value'] = olympic_clean['Medal'].map(medal_map).fillna(0)

print("Medal distribution:")
print(olympic_clean['Medal'].value_counts(dropna=False))

print("\nMedal won distribution:")
print(olympic_clean['Medal_Won'].value_counts())

print(f"\nTotal records: {len(olympic_clean)}")

Medal distribution:
Medal
NaN       231333
Gold       13372
Bronze     13295
Silver     13116
Name: count, dtype: int64

Medal won distribution:
Medal_Won
False    231333
True      39783
Name: count, dtype: int64

Total records: 271116


In [4]:
# Merge athletes with NOC regions to get country names
# This is a left join - keep all Olympic records
olympic_with_country = olympic_clean.merge(
    noc_map,
    on='NOC',
    how='left'
)

# Check the merge results
print(f"Records before merge: {len(olympic_clean)}")
print(f"Records after merge: {len(olympic_with_country)}")

# Check how many NOC codes matched
matched = olympic_with_country['region'].notna().sum()
total = len(olympic_with_country)
print(f"\nMatched records: {matched} out of {total} ({matched/total*100:.1f}%)")

# See which NOC codes didn't match
unmatched_nocs = olympic_with_country[olympic_with_country['region'].isna()]['NOC'].unique()
print(f"\nUnmatched NOC codes ({len(unmatched_nocs)}): {sorted(unmatched_nocs)}")

print("\n=== First few rows after merge ===")
print(olympic_with_country[['Name', 'NOC', 'region', 'Year', 'Sport', 'Medal']].head())

Records before merge: 271116
Records after merge: 271116

Matched records: 270746 out of 271116 (99.9%)

Unmatched NOC codes (4): ['ROT', 'SGP', 'TUV', 'UNK']

=== First few rows after merge ===
                       Name  NOC       region  Year          Sport Medal
0                 A Dijiang  CHN        China  1992     Basketball   NaN
1                  A Lamusi  CHN        China  2012           Judo   NaN
2       Gunnar Nielsen Aaby  DEN      Denmark  1920       Football   NaN
3      Edgar Lindenau Aabye  DEN      Denmark  1900     Tug-Of-War  Gold
4  Christine Jacoba Aaftink  NED  Netherlands  1988  Speed Skating   NaN


In [5]:
# The GDP data is in wide format with years as columns
# We need to convert it to long format with Year and GDP_per_capita columns

# First, keep only the columns we need
# Drop 'Indicator Name' and 'Indicator Code' as they're the same for all rows
gdp_subset = gdp_raw.drop(['Indicator Name', 'Indicator Code'], axis=1)

# Reshape from wide to long format
# This converts year columns (1960, 1961, etc.) into rows
gdp_long = gdp_subset.melt(
    id_vars=['Country Name', 'Country Code'],
    var_name='Year',
    value_name='GDP_per_capita'
)

# Convert Year to integer
gdp_long['Year'] = pd.to_numeric(gdp_long['Year'], errors='coerce')

# Remove rows where Year conversion failed or GDP is missing
gdp_long = gdp_long.dropna(subset=['Year', 'GDP_per_capita'])
gdp_long['Year'] = gdp_long['Year'].astype(int)

print(f"GDP data reshaped: {len(gdp_long)} country-year records")
print(f"Year range: {gdp_long['Year'].min()} to {gdp_long['Year'].max()}")
print(f"Number of unique countries: {gdp_long['Country Code'].nunique()}")

print("\n=== First few rows of reshaped GDP data ===")
print(gdp_long.head(10))

GDP data reshaped: 14561 country-year records
Year range: 1960 to 2024
Number of unique countries: 262

=== First few rows of reshaped GDP data ===
                   Country Name Country Code  Year  GDP_per_capita
1   Africa Eastern and Southern          AFE  1960      186.089204
3    Africa Western and Central          AFW  1960      121.936832
9                     Argentina          ARG  1960      778.251707
13                    Australia          AUS  1960     1813.431099
14                      Austria          AUT  1960      939.914815
16                      Burundi          BDI  1960       70.905100
17                      Belgium          BEL  1960     1290.286072
18                        Benin          BEN  1960       89.856925
19                 Burkina Faso          BFA  1960       69.150246
20                   Bangladesh          BGD  1960       82.481277


In [6]:
# World Bank includes regional aggregates (like 'Africa Eastern and Southern')
# We only want actual countries that participate in Olympics

# Get the list of unique NOC codes from our Olympic data
olympic_countries = olympic_with_country['NOC'].unique()

# We need to create a mapping from NOC to Country Code
# For now, let's filter GDP data to remove obvious regional aggregates
# Regional codes are usually longer or contain specific patterns

# Get unique country codes from GDP data
gdp_countries = gdp_long['Country Code'].unique()

print(f"Countries in GDP data: {len(gdp_countries)}")
print(f"Sample GDP country codes: {sorted(gdp_countries)[:20]}")

# Remove regional aggregates - they typically have codes like AFE, AFW, etc.
# Keep only 3-letter codes that look like country codes
# We'll do a more sophisticated filter in the next cell
regional_codes = ['AFE', 'AFW', 'ARB', 'CEB', 'CSS', 'EAP', 'EAR', 'EAS', 'ECA', 
                  'ECS', 'EMU', 'EUU', 'FCS', 'HIC', 'HPC', 'IBD', 'IBT', 'IDA',
                  'IDB', 'IDX', 'INX', 'LAC', 'LCN', 'LDC', 'LIC', 'LMC', 'LMY',
                  'LTE', 'MEA', 'MIC', 'MNA', 'NAC', 'OED', 'OSS', 'PRE', 'PSS',
                  'PST', 'SAS', 'SSA', 'SSF', 'SST', 'TEA', 'TEC', 'TLA', 'TMN',
                  'TSA', 'TSS', 'UMC', 'WLD']

gdp_countries_only = gdp_long[~gdp_long['Country Code'].isin(regional_codes)].copy()

print(f"\nAfter removing regional aggregates: {len(gdp_countries_only)} records")
print(f"Unique countries remaining: {gdp_countries_only['Country Code'].nunique()}")

Countries in GDP data: 262
Sample GDP country codes: ['ABW', 'AFE', 'AFG', 'AFW', 'AGO', 'ALB', 'AND', 'ARB', 'ARE', 'ARG', 'ARM', 'ASM', 'ATG', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA']

After removing regional aggregates: 11569 records
Unique countries remaining: 214


In [7]:
# We need to map NOC codes to ISO country codes for merging
# Some match directly (CHN->CHN, USA->USA) but many don't (DEN->DNK, GER->DEU)

# Create a manual mapping for common mismatches
# This is based on standard NOC to ISO mappings
noc_to_iso = {
    'DEN': 'DNK',  # Denmark
    'GER': 'DEU',  # Germany
    'SUI': 'CHE',  # Switzerland
    'NED': 'NLD',  # Netherlands
    'GRE': 'GRC',  # Greece
    'POR': 'PRT',  # Portugal
    'ESP': 'ESP',  # Spain (same)
    'SWE': 'SWE',  # Sweden (same)
    'BEL': 'BEL',  # Belgium (same)
    'AUT': 'AUT',  # Austria (same)
    'NOR': 'NOR',  # Norway (same)
    'FIN': 'FIN',  # Finland (same)
    'ITA': 'ITA',  # Italy (same)
    'FRA': 'FRA',  # France (same)
    'GBR': 'GBR',  # Great Britain (same)
    'USA': 'USA',  # United States (same)
    'CAN': 'CAN',  # Canada (same)
    'AUS': 'AUS',  # Australia (same)
    'NZL': 'NZL',  # New Zealand (same)
    'JPN': 'JPN',  # Japan (same)
    'CHN': 'CHN',  # China (same)
    'KOR': 'KOR',  # South Korea (same)
    'BRA': 'BRA',  # Brazil (same)
    'ARG': 'ARG',  # Argentina (same)
    'MEX': 'MEX',  # Mexico (same)
    'RSA': 'ZAF',  # South Africa
    'IND': 'IND',  # India (same)
    'RUS': 'RUS',  # Russia (same)
    'POL': 'POL',  # Poland (same)
    'HUN': 'HUN',  # Hungary (same)
    'CZE': 'CZE',  # Czech Republic (same)
    'ROU': 'ROU',  # Romania (same)
    'BUL': 'BGR',  # Bulgaria
    'TUR': 'TUR',  # Turkey (same)
    'EGY': 'EGY',  # Egypt (same)
    'KEN': 'KEN',  # Kenya (same)
    'ETH': 'ETH',  # Ethiopia (same)
    'JAM': 'JAM',  # Jamaica (same)
    'CUB': 'CUB',  # Cuba (same)
    'UKR': 'UKR',  # Ukraine (same)
}

# Add the ISO country code to our Olympic data
olympic_with_country['ISO_Code'] = olympic_with_country['NOC'].map(noc_to_iso)

# For NOC codes that match ISO directly, use the NOC code
olympic_with_country['ISO_Code'] = olympic_with_country['ISO_Code'].fillna(
    olympic_with_country['NOC']
)

print("Sample NOC to ISO mapping:")
print(olympic_with_country[['NOC', 'region', 'ISO_Code']].drop_duplicates().head(20))

# Check how many unique ISO codes we have
print(f"\nUnique ISO codes: {olympic_with_country['ISO_Code'].nunique()}")

Sample NOC to ISO mapping:
     NOC       region ISO_Code
0    CHN        China      CHN
2    DEN      Denmark      DNK
4    NED  Netherlands      NLD
10   USA          USA      USA
28   FIN      Finland      FIN
59   NOR       Norway      NOR
80   ROU      Romania      ROU
94   EST      Estonia      EST
98   FRA       France      FRA
134  MAR      Morocco      MAR
139  ESP        Spain      ESP
146  EGY        Egypt      EGY
147  IRI         Iran      IRI
151  BUL     Bulgaria      BGR
158  ITA        Italy      ITA
160  CHA         Chad      CHA
161  AZE   Azerbaijan      AZE
162  SUD        Sudan      SUD
163  RUS       Russia      RUS
165  ARG    Argentina      ARG

Unique ISO codes: 230


In [8]:
# Now merge the Olympic data with GDP data
# Match on both ISO_Code (Country Code) and Year

merged_data = olympic_with_country.merge(
    gdp_countries_only[['Country Code', 'Year', 'GDP_per_capita']],
    left_on=['ISO_Code', 'Year'],
    right_on=['Country Code', 'Year'],
    how='left'
)

# Check merge results
print(f"Total Olympic records: {len(olympic_with_country)}")
print(f"Records after GDP merge: {len(merged_data)}")

# Check how many records got GDP data
gdp_matched = merged_data['GDP_per_capita'].notna().sum()
print(f"\nRecords with GDP data: {gdp_matched} ({gdp_matched/len(merged_data)*100:.1f}%)")

print("\nGDP Missing AFTER 1960:")
post_1960 = merged_data[merged_data['Year'] >= 1960]
missing_post_1960 = post_1960['GDP_per_capita'].isna().mean() * 100
print(f"{missing_post_1960:.2f}% of post-1960 records missing GDP")

# Analyze coverage by year
print("\n=== GDP Coverage by Year ===")
coverage_by_year = merged_data.groupby('Year').agg({
    'GDP_per_capita': lambda x: x.notna().sum(),
    'ID': 'count'
})
coverage_by_year.columns = ['Records_with_GDP', 'Total_Records']
coverage_by_year['Coverage_%'] = (coverage_by_year['Records_with_GDP'] / 
                                   coverage_by_year['Total_Records'] * 100).round(1)

print(coverage_by_year.tail(20))

Total Olympic records: 271116
Records after GDP merge: 271116

Records with GDP data: 167556 (61.8%)

GDP Missing AFTER 1960:
19.86% of post-1960 records missing GDP

=== GDP Coverage by Year ===
      Records_with_GDP  Total_Records  Coverage_%
Year                                             
1964              6560           9480        69.2
1968              6580          10479        62.8
1972              7567          11959        63.3
1976              6622          10502        63.1
1980              5150           8937        57.6
1984              8791          11588        75.9
1988             10342          14676        70.5
1992             12994          16413        79.2
1994              2994           3160        94.7
1996             12278          13780        89.1
1998              3418           3605        94.8
2000             12379          13821        89.6
2002              3871           4109        94.2
2004             12105          13443        90.0
2006

In [9]:
# Select and rename columns for the final dataset
final_columns = {
    'ID': 'Athlete_ID',
    'Name': 'Athlete_Name',
    'Sex': 'Sex',
    'Age': 'Age',
    'Height': 'Height_cm',
    'Weight': 'Weight_kg',
    'Team': 'Team',
    'NOC': 'NOC',
    'region': 'Country_Name',
    'ISO_Code': 'Country_Code',
    'Games': 'Games',
    'Year': 'Year',
    'Season': 'Season',
    'City': 'Host_City',
    'Sport': 'Sport',
    'Event': 'Event',
    'Medal': 'Medal',
    'Medal_Won': 'Medal_Won',
    'Medal_Value': 'Medal_Value',
    'GDP_per_capita': 'GDP_per_capita'
}

merged_final = merged_data[list(final_columns.keys())].copy()
merged_final.columns = list(final_columns.values())

# Drop the duplicate Country_Code column that came from GDP merge
if 'Country Code' in merged_data.columns:
    pass  # Already handled in column selection

print("=== Final Merged Dataset ===")
print(f"Total records: {len(merged_final)}")
print(f"Total columns: {len(merged_final.columns)}")
print(f"\nColumn names:")
print(list(merged_final.columns))

print("\n=== Sample of final data ===")
print(merged_final.head(10))

print("\n=== Data types ===")
print(merged_final.dtypes)

=== Final Merged Dataset ===
Total records: 271116
Total columns: 20

Column names:
['Athlete_ID', 'Athlete_Name', 'Sex', 'Age', 'Height_cm', 'Weight_kg', 'Team', 'NOC', 'Country_Name', 'Country_Code', 'Games', 'Year', 'Season', 'Host_City', 'Sport', 'Event', 'Medal', 'Medal_Won', 'Medal_Value', 'GDP_per_capita']

=== Sample of final data ===
   Athlete_ID              Athlete_Name Sex   Age  Height_cm  Weight_kg  \
0           1                 A Dijiang   M  24.0      180.0       80.0   
1           2                  A Lamusi   M  23.0      170.0       60.0   
2           3       Gunnar Nielsen Aaby   M  24.0        NaN        NaN   
3           4      Edgar Lindenau Aabye   M  34.0        NaN        NaN   
4           5  Christine Jacoba Aaftink   F  21.0      185.0       82.0   
5           5  Christine Jacoba Aaftink   F  21.0      185.0       82.0   
6           5  Christine Jacoba Aaftink   F  25.0      185.0       82.0   
7           5  Christine Jacoba Aaftink   F  25.0      

In [10]:
# Data Cleaning + Validating

print("="*60)
print("DATA QUALITY SUMMARY REPORT")
print("="*60)

print("\n1. DATASET SIZE")
print(f"   Total athlete-event records: {len(merged_final):,}")
print(f"   Unique athletes: {merged_final['Athlete_ID'].nunique():,}")
print(f"   Unique countries: {merged_final['Country_Code'].nunique()}")
print(f"   Year range: {merged_final['Year'].min()} - {merged_final['Year'].max()}")

print("\n2. MISSING VALUES")
missing = merged_final.isnull().sum()
missing_pct = (missing / len(merged_final) * 100).round(1)
missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Percent': missing_pct
})
print(missing_df[missing_df['Missing_Count'] > 0])

print("\n3. MEDAL DISTRIBUTION")
print(merged_final['Medal'].value_counts(dropna=False))

print("\n4. GDP COVERAGE BY SEASON")
summer = merged_final[merged_final['Season'] == 'Summer']
winter = merged_final[merged_final['Season'] == 'Winter']
print(f"   Summer Olympics - GDP coverage: {summer['GDP_per_capita'].notna().sum() / len(summer) * 100:.1f}%")
print(f"   Winter Olympics - GDP coverage: {winter['GDP_per_capita'].notna().sum() / len(winter) * 100:.1f}%")

print("\n5. TOP 10 COUNTRIES BY TOTAL RECORDS")
print(merged_final['Country_Name'].value_counts().head(10))

print("\n6. GDP STATISTICS (for records with GDP data)")
gdp_stats = merged_final[merged_final['GDP_per_capita'].notna()]['GDP_per_capita']
print(f"   Mean GDP per capita: ${gdp_stats.mean():,.2f}")
print(f"   Median GDP per capita: ${gdp_stats.median():,.2f}")
print(f"   Min GDP per capita: ${gdp_stats.min():,.2f}")
print(f"   Max GDP per capita: ${gdp_stats.max():,.2f}")

DATA QUALITY SUMMARY REPORT

1. DATASET SIZE
   Total athlete-event records: 271,116
   Unique athletes: 135,571
   Unique countries: 230
   Year range: 1896 - 2016

2. MISSING VALUES
                Missing_Count  Missing_Percent
Age                      9474              3.5
Height_cm               60171             22.2
Weight_kg               62875             23.2
Country_Name              370              0.1
Medal                  231333             85.3
GDP_per_capita         103560             38.2

3. MEDAL DISTRIBUTION
Medal
NaN       231333
Gold       13372
Bronze     13295
Silver     13116
Name: count, dtype: int64

4. GDP COVERAGE BY SEASON
   Summer Olympics - GDP coverage: 59.1%
   Winter Olympics - GDP coverage: 74.4%

5. TOP 10 COUNTRIES BY TOTAL RECORDS
Country_Name
USA          18853
Germany      15883
France       12758
UK           12256
Russia       11692
Italy        10715
Canada        9734
Japan         8444
Sweden        8339
Australia     7724
Name: count, d

In [11]:
# Save the merged dataset to the processed data folder
output_path = '../data/processed/'

# Create the processed folder if it doesn't exist
os.makedirs(output_path, exist_ok=True)

# Save as CSV
output_file = output_path + 'olympics_gdp_merged.csv'
merged_final.to_csv(output_file, index=False)

print(f"Merged dataset saved successfully!")
print(f"Location: {output_file}")
print(f"File size: {os.path.getsize(output_file) / (1024*1024):.2f} MB")

print("\n=== File saved with the following structure ===")
print(f"Rows: {len(merged_final)}")
print(f"Columns: {len(merged_final.columns)}")
print(f"\nYou can now use this file for EDA in the next notebook!")

Merged dataset saved successfully!
Location: ../data/processed/olympics_gdp_merged.csv
File size: 43.48 MB

=== File saved with the following structure ===
Rows: 271116
Columns: 20

You can now use this file for EDA in the next notebook!
